# 3. EMA Backtesting

This notebook __backtests__ the EMA touch-and-rejection strategy from backtest.py via two routes:
1. **Route A — manual EMAs**: backtest specific EMAs you chose from notebook 2's rankings.
2. **Route B — walk-forward + auto picker**: split data 70/30, let pick_best_ema_* choose on the train slice, evaluate on the held-out test slice.

Plus parameter sweeps over stop loss, target, EMA period, regime filter, and direction (works on either route's cfg).

**Prerequisite**:
- run 1_core_pipeline.ipynb first to populate the OHLC cache and configuration.
- notebook 2 is optional but recommended for Route A — that's where you find the EMA candidates worth backtesting.

## Table of contents

1. [Setup](#Setup)
2. [Load pipeline state](#Load-pipeline-state)
3. [Filter thresholds](#Filter-thresholds)
4. [Backtesting](#Backtesting)
    1. [Setup](#Setup)
    2. [Capital, sizing, fees, slippage](#Capital,-sizing,-fees,-slippage)
    3. [Route A — Manual EMAs from notebook 2](#Route-A-—-Manual-EMAs-from-notebook-2)
    4. [Route B — Walk-forward + auto picker](#Route-B-—-Walk-forward-+-auto-picker)
        1. [Strategy configuration](#Strategy-configuration)
        2. [Run on train (in-sample)](#Run-on-train-(in-sample))
        3. [Run on test (out-of-sample)](#Run-on-test-(out-of-sample))
        4. [Reading the result](#Reading-the-result)
    5. [Parameter sweep: stop × target](#Parameter-sweep:-stop-×-target)
    6. [Full sweep: stop × target × EMA × regime × direction](#Full-sweep:-stop-×-target-×-EMA-×-regime-×-direction)

## Setup


In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

from ema_core import (
    load_and_analyze,
    filter_by_cross_rate,
    pick_best_ema_support, pick_best_ema_resistance, pick_best_ema_universal,
)
from backtest import (
    StrategyConfig, StopLossConfig, TakeProfitConfig, PositionSizingConfig,
    backtest, compute_metrics, print_metrics,
    plot_equity_curve, plot_trades_on_chart,
    walk_forward_split, sweep_configs, sweep_grid,
)


## Load pipeline state

load_and_analyze reads `data/config.json` (written by 1_core_pipeline.ipynb), loads OHLC from the parquet cache (or fetches if missing), and runs analyze_ema_touches. Returns df, result, cfg in one call.

To change any of these values, edit them in notebook 1's config cell and re-run it — this notebook will pick up the new values on its next run.

In [ ]:
# Load OHLC from cache (or fetch) + run the per-EMA analysis. Single source of truth: data/config.json.
df, result, cfg = load_and_analyze()

# Unpack config values used by the backtest cells below (delta, delta_mode, ema_range).
symbol     = cfg["symbol"]
interval   = cfg["interval"]
start      = cfg["start"]
end        = cfg["end"]
ema_range  = cfg["ema_range"]
delta      = cfg["delta"]
delta_mode = cfg["delta_mode"]
category   = cfg["category"]

print(f"  -> {symbol} {interval} | {start} → {end} | delta={delta} ({delta_mode}) | category={category}")
print(f"     df: {len(df)} candles | result: {len(result)} EMAs")


## Filter thresholds

MIN_TOUCHES_* are computed from the full result. They're passed explicitly to pick_best_ema_* further down.

In [ ]:
MAX_CROSS_RATE  = 0.3
MIN_TOUCHES_SUP = int(result["support_test"].median())
MIN_TOUCHES_RES = int(result["resistance_test"].median())
MIN_TOUCHES_UNI = int((result["any_touch"] + result["cross"]).median())
print(f"MIN_TOUCHES_SUP={MIN_TOUCHES_SUP} | MIN_TOUCHES_RES={MIN_TOUCHES_RES} | MIN_TOUCHES_UNI={MIN_TOUCHES_UNI}")


## Backtesting

Translates the EMA S/R analysis into a fully simulated trading strategy with:
- realistic execution: fees + slippage
- configurable: stop loss, take profit, position sizing
- per-trade metrics: expectancy, profit factor, max drawdown, exit-reason breakdown, etc.

__Strategy logic:__
- Entry timing: at the close of the touch candle, with slippage applied.
- Long Entry: a candle's Low touches a chosen support EMA (within delta) and the candle closes back above the EMA (rejection). \
  Optionally gated by close > EMA(regime_filter).
- Short entry: a candle's High touches a chosen resistance EMA (within delta) and the candle closes back below the EMA (rejection). \
  Optionally gated by close < EMA(regime_filter).
- One position at a time.
- Exit: stop loss, take profit, or end-of-data — whichever fires first. \
  If both SL and TP are reachable in the same candle, SL is assumed to fire first (conservative).

__Optional regime_filter:__
- adds a broader trend check
- it is a second, optional entry condition that has to also be true for an entry to fire

The default entry signal is local — it only sees the bar near the entry EMA, not the broader trend. \
regime_filter=100 adds a second slower EMA gate: longs only fire when close is also above EMA 100. \
Meaning: Buy pullbacks only when the bigger picture agrees — price still above the slow EMA. \
Mirror logic for shorts: close < EMA(regime_filter) — you only short rallies when the broader trend is also down.

*Example:*
1. Daily: BTC in a long downtrend.
2. 15-min: a counter-trend rally pushes price above EMA 20.
3. After the rally stalls, price retraces down toward EMA 20 — bar's low taps EMA 20, close holds at/above it.
4. Long signal fires (without regime filter).
5. Daily downtrend resumes, price tanks below EMA 20, the long is stopped out.
6. With regime_filter=100: close < EMA 100 (price is still below the slower trend EMA), so step 4 is blocked.

__*Rule of thumb: regime_filter should be slower than ema_period*__

The whole point of the filter is to use a slower, broader-trend EMA to gate entries on the faster, entry EMA. \
Common sensible pairings:

| ema_period (entry) | regime_filter (trend) | what it captures |
|---|---|---|
| 20 | 50  | light filter — intraday entry, daily trend |
| 20 | 100 | tighter — needs the medium-term trend agreement |
| 50 | 200 | classic "above EMA 200 = bull regime" |

__Configurable knobs:__
- direction:
  - long
  - short
  - both
- ema_period:
  - int
  - best EMA with the highest ratio
- delta:
  - int
- delta_mode:
  - percent (in % of the EMA value)
  - absolute (fixed, in quote currency)
- stop_loss.mode:
  - fixed_value
  - fixed_pct
  - beyond_ema
  - trailing
  - chandelier
- take_profit.mode:
  - fixed_value
  - fixed_pct
  - r_multiple
  - trailing
- position_sizing.mode:
  - fixed_value (notional in quote currency)
  - fixed_pct (% of equity)
- fee_pct: 
  - per-fill, in % (e.g. 0.055 for ByBit taker)
- slippage_pct: 
  - per-fill, in % of price
- initial_capital:
  - starting equity in quote currency (e.g. 10_000 for $10k)
- regime_filter:
  - None (default)
  - int N (longs only when close > EMA(N))
  - int N (shorts only when close < EMA(N))

__Two routes from here, depending on your goal:__

- **Route A — manual EMAs**: backtest specific EMAs you chose from notebook 2's rankings. Cleanest path for validating hand-picked candidates and comparing them head-to-head.
- **Route B — walk-forward + auto picker**: split data 70/30, let pick_best_ema_* choose on the train slice, evaluate on the held-out test slice. Strict OOS validation.

Both routes assemble a cfg (StrategyConfig) you can pass to the parameter sweeps further down. They're independent — run whichever fits your goal, or run both (the second one overwrites cfg).

The two cells below (Setup, Capital/sizing/fees) are shared and apply to both routes.

### Setup

Single import + autoreload so edits to backtest.py hot-reload without restarting the kernel.

In [57]:
# Engine + visualisation lives in backtest.py — import once and call from the cells below.
# %autoreload picks up edits to backtest.py without restarting the kernel.
# You only need to run them once per kernel session. Subsequent edits to backtest.py will pick up automatically without re-importing or restarting.

%load_ext autoreload
%autoreload 2

from dataclasses import replace
from backtest import (
    StopLossConfig, TakeProfitConfig, PositionSizingConfig, StrategyConfig,
    backtest, compute_metrics, print_metrics,
    plot_equity_curve, plot_trades_on_chart,
    walk_forward_split,
    sweep_configs, sweep_grid,
)

### Capital, sizing, fees, slippage

__Sizing & capital — what to set, and why it matters.__

The position sizer turns a notional dollar amount into a coin quantity. \
ByBit's BTCUSDT minimum is __0.001 BTC__, so any trade whose computed size falls below that gets skipped (the engine prints a warning the first time and a count at the end of the run).

To clear the floor, your per-trade notional must be ≥ min_size × price. \
At BTC ≈ ＄70k that's ≈ ＄70 per trade; at ＄100k it's ≈ ＄100. \
So:

- position_sizing=fixed_pct(10) on initial_capital=10_000 → ＄1000/trade. \
  Comfortable across all BTC price levels and through deep drawdowns.
- position_sizing=fixed_pct(10) on initial_capital=1_000 → ＄100/trade. \
  Works at BTC ≤ ≈ ＄100k, but a >30% drawdown shrinks notional below the floor and trades start getting silently skipped.
- position_sizing=fixed_pct(10) on initial_capital=100 → ＄10/trade. \
  Always below the floor at any plausible BTC price → __zero trades__.
- position_sizing=fixed_value(100) → flat ＄100/trade regardless of equity. \
  Cleanest way to test on small capital; tradeoff is no compounding on the upside and no automatic de-risking on the downside.

__Trading costs matter more on small capital.__

ByBit taker fee 0.055% × 2 fills + slippage = ≈ 0.13% per round-trip. \
On a ＄100 notional that's ≈ ＄0.13/trade — same in absolute terms regardless of equity, but a larger fraction of a smaller account.

__Rule of thumb:__
- Pick capital and sizing such that even a 30–50% drawdown still produces notional ≥ min_size × max_expected_price. \
- If you want to backtest realistically on a small account, use fixed_value sizing.

### Route A — Manual EMAs from notebook 2

Use this route when you have specific EMA candidates in mind — typically the top-ranked EMAs from notebook 2's filtered ratio tables (ratio_support_1_filtered for longs, ratio_resistance_1_filtered for shorts, ratio_universal_1_filtered for both).

What you get:
1. A leaderboard comparing your candidates head-to-head on the full df.
2. The option to drill into one candidate with equity curve + trade chart.

__Note on interpretation:__ \
You chose these EMAs using information from the full period (notebook 2 ran on the full df), so the result is "in-sample with respect to your selection." The numbers tell you how these EMAs traded on this data — they don't claim out-of-sample edge. Use them to (a) confirm the S/R analysis translated to actual P&L, and (b) compare candidates head-to-head. For strict OOS, jump to Route B.

In [ ]:
# Edit this list — typically the top EMAs from notebook 2's filtered ratio tables
# (ratio_support_1_filtered for longs, ratio_resistance_1_filtered for shorts,
#  ratio_universal_1_filtered for "both").
candidates = [77, 84, 114]
direction  = "long"   # "long" / "short" / "both"

# Run a backtest per candidate on the full df, keep the artifacts so we can
# drill into any one of them without re-running.
backtests = {}    # ema -> (cfg, trades, equity, metrics)
rows = []
for ema in candidates:
    cfg_i = StrategyConfig(
        direction=direction,
        ema_period=ema,
        delta=delta,
        delta_mode=delta_mode,
        stop_loss=StopLossConfig(mode="fixed_pct", value=1.0),         # 1% stop
        take_profit=TakeProfitConfig(mode="r_multiple", value=3.0),    # 3R target
        position_sizing=PositionSizingConfig(mode="fixed_value", value=100.0),
        fee_pct=0.055,        # ByBit taker fee
        slippage_pct=0.01,    # 0.01% of price
        initial_capital=100.0,
        regime_filter=None,
    )
    trades_i, equity_i = backtest(df, cfg_i)
    m_i = compute_metrics(trades_i, equity_i, cfg_i.initial_capital)
    backtests[ema] = (cfg_i, trades_i, equity_i, m_i)
    rows.append({
        "ema":              ema,
        "trades":           m_i.get("trades", 0),
        "win_rate_pct":     round(m_i.get("win_rate", 0) * 100, 2),
        "expectancy":       m_i.get("expectancy_per_trade", 0),
        "profit_factor":    m_i.get("profit_factor", 0),
        "total_return_pct": m_i.get("total_return_pct", 0),
        "max_dd_pct":       m_i.get("max_drawdown_pct", 0),
    })

leaderboard = pd.DataFrame(rows).sort_values("expectancy", ascending=False).reset_index(drop=True)
leaderboard

__How to read the leaderboard:__
- Sort by **expectancy** for absolute per-trade dollar edge; by **profit_factor** for risk-adjusted edge.
- Cross-check **trades**: a top-ranked EMA with <20 trades is a noisy estimate. Prefer candidates with enough samples to trust the number.
- If multiple notebook-2 picks land in positive expectancy here, the S/R analysis is producing tradeable signal — drill into one below.
- If everything is negative, the touch-and-rejection rule isn't extracting edge from these levels on this data; widen stops, lower R-targets, or move to Route B's walk-forward sanity check.

__Drill into one candidate__

In [ ]:
# Pick one EMA from the leaderboard to drill into. Default: the top-ranked candidate.
chosen_ema = int(leaderboard.iloc[0]["ema"])

# Reuse the cached backtest from the loop above — no need to re-run.
cfg, trades_a, equity_a, _ = backtests[chosen_ema]
print(f"Inspecting EMA {chosen_ema} ({direction}):")
print_metrics(compute_metrics(trades_a, equity_a, cfg.initial_capital))

In [ ]:
plot_equity_curve(equity_a, cfg.initial_capital, title=f"Equity curve — Route A (EMA {cfg.ema_period})")

In [ ]:
plot_trades_on_chart(df, trades_a, cfg, initial_window_days=30)

### Route B — Walk-forward + auto picker

Use this route when you want strict out-of-sample validation:

1. Split the data 70/30 — the first 70% is **train** (where the picker fits), the last 30% is **test** (held out, the picker never sees it).
2. pick_best_ema_* (matching the strategy direction) selects the EMA with the highest hold rate on train.
3. The same StrategyConfig runs on train (in-sample, optimistic by construction) and on test (out-of-sample, the honest read).
4. If train and test agree, there's a candidate edge. If test collapses while train looked great, the EMA selection over-fit.

Anything refitted on test is overfit by definition.

The default call below uses the configured filter globals (MIN_TOUCHES_* and MAX_CROSS_RATE) and the ema_range from `data/config.json`. Override only if you want different filter strictness on the train slice — e.g.:

```python
best_ema = pick_best_ema_support(train_df, ema_range=range(20, 200, 10),
                                 delta=delta, delta_mode=delta_mode,
                                 min_touches=20, max_cross_rate=0.3)
```

Changing direction at the top of the cell below auto-propagates to both the picker (which selects the EMA) and the StrategyConfig further down. Re-run from this cell to apply.

In [ ]:
# Reuse df already fetched above.
# Default config: 1% fixed stop loss, 3R take-profit, 10% of equity per trade, 0.055% ByBit taker fee, 0.01% slippage.
# EMA chosen via the appropriate ratio (support / resistance / universal) on a train slice.

# Choose direction here — same value should be used in StrategyConfig below.
direction = "long"   # "long" / "short" / "both"

train_df, test_df = walk_forward_split(df, train_pct=0.7)
print(f"Train: {train_df['timestamp'].iloc[0]} → {train_df['timestamp'].iloc[-1]}  ({len(train_df)} candles)")
print(f"Test:  {test_df['timestamp'].iloc[0]} → {test_df['timestamp'].iloc[-1]}  ({len(test_df)} candles)")

# Pick the EMA whose evaluation criterion matches the strategy direction.
# All filter thresholds come from the notebook config (MAX_CROSS_RATE, MIN_TOUCHES_*).
if direction == "long":
    best_ema = pick_best_ema_support(train_df, ema_range=ema_range,
                                     delta=delta, delta_mode=delta_mode,
                                     min_touches=MIN_TOUCHES_SUP,
                                     max_cross_rate=MAX_CROSS_RATE)
elif direction == "short":
    best_ema = pick_best_ema_resistance(train_df, ema_range=ema_range,
                                        delta=delta, delta_mode=delta_mode,
                                        min_touches=MIN_TOUCHES_RES,
                                        max_cross_rate=MAX_CROSS_RATE)
else:   # "both"
    best_ema = pick_best_ema_universal(train_df, ema_range=ema_range,
                                       delta=delta, delta_mode=delta_mode,
                                       min_touches=MIN_TOUCHES_UNI,
                                       max_cross_rate=MAX_CROSS_RATE)

print(f"Best {direction} EMA on train: {best_ema}")

#### Strategy configuration

Assemble the StrategyConfig from the sizing & fee choices above and the direction & EMA picked on train.

In [ ]:
cfg = StrategyConfig(
    direction=direction,            # set in the walk-forward cell above
    ema_period=best_ema,            # picked on train; replace with an int to override manually
    delta=delta,
    delta_mode=delta_mode,
    stop_loss=StopLossConfig(mode="fixed_pct", value=1.0),         # 1% stop
    take_profit=TakeProfitConfig(mode="r_multiple", value=3.0),    # 3R target
    position_sizing=PositionSizingConfig(mode="fixed_value", value=100.0),      # fixed_pct is % of equity per trade
    fee_pct=0.055,        # ByBit taker fee
    slippage_pct=0.01,    # 0.01% of price
    initial_capital=100.0,
    regime_filter=None,
)
cfg

#### Run on train (in-sample)

Fit the strategy to the training slice. The numbers below will look optimistic — that's expected; the test slice (next section) is the honest read.

In [62]:
trades_train, equity_train = backtest(train_df, cfg)
metrics_train = compute_metrics(trades_train, equity_train, cfg.initial_capital)
print_metrics(metrics_train)

Trades:               34  (8W / 26L)
Win rate:             23.53%
Avg win:                    2.88
Avg loss:                  -1.12
Expectancy per trade:      -0.18
Profit factor:        0.791
Total return:         -6.08%
Final equity:              93.92
Max drawdown:         -13.46%
Sharpe (per-trade):   -0.634
Sortino (per-trade):  0.000
Total fees paid:      3.74
Avg bars held:        66.1
Exit reasons:         {'stop_loss': 26, 'take_profit': 8}


Equity curve and drawdown over time.

In [ ]:
plot_equity_curve(equity_train, cfg.initial_capital, title=f"Equity curve — train (EMA {cfg.ema_period})")

Trades on the price chart — blue ▲ longs, red ▼ shorts, ✕ exits color-coded by reason.

In [ ]:
plot_trades_on_chart(train_df, trades_train, cfg, initial_window_days=30)

#### Run on test (out-of-sample)

Same config on data the EMA picker never saw. If train and test agree (both positive after fees), there's a candidate edge. If test collapses while train looked great, the EMA selection over-fit.

In [63]:
trades_test, equity_test = backtest(test_df, cfg)
metrics_test = compute_metrics(trades_test, equity_test, cfg.initial_capital)
print_metrics(metrics_test)

Trades:               18  (3W / 15L)
Win rate:             16.67%
Avg win:                    2.88
Avg loss:                  -1.12
Expectancy per trade:      -0.45
Profit factor:        0.514
Total return:         -8.16%
Final equity:              91.84
Max drawdown:         -9.32%
Sharpe (per-trade):   -3.553
Sortino (per-trade):  0.000
Total fees paid:      1.98
Avg bars held:        40.5
Exit reasons:         {'stop_loss': 15, 'take_profit': 3}


Equity curve and drawdown over time.

In [ ]:
plot_equity_curve(equity_test, cfg.initial_capital, title=f"Equity curve — test (EMA {cfg.ema_period})")

Trades on the price chart — blue ▲ longs, red ▼ shorts, ✕ exits color-coded by reason.

In [ ]:
plot_trades_on_chart(test_df, trades_test, cfg, initial_window_days=30)

#### Reading the result

- If __test expectancy > 0 after fees__ and is in the same ballpark as train → there's a real edge worth paper-trading next.
- If __test expectancy < 0__ while train was positive → the EMA selection over-fit the train slice. Try a coarser ema_range, a longer history, or different delta.
- If __profit factor < 1.2__ on test → edge is too thin to survive real-world frictions you didn't model (funding, partial fills, exchange downtime).
- __Exit reason mix__ tells you whether the 3R target is realistic. If stop_loss >> take_profit, you're getting stopped out before the move develops — try wider stops, lower R-targets, or a slower EMA.

__Next iterations__ (separate experiments, not this run):
1. Sweep EMA period × stop% × R-target on train, pick top 3 by Sortino, validate each on test.
2. Sweep direction (long / short / both). In a chop-heavy regime both typically beats either side alone because mean-reversion fires on both swings; in strong trends, the trending direction wins.
3. Add a regime filter (e.g. only trade longs when price is above a slow EMA like 200) to skip counter-trend stretches.
4. Replace single-asset with a basket (BTC + ETH + a couple of liquid alts) — diversifies the edge.

Train and test agree → not curve-fit, just a non-edge as configured. \
The 3R target only fires 22% of the time, and 1% stops absorb the rest.

Variations to sweep next: 
- wider stops (1.5–2%), 
- lower R-targets (1.5–2R), 
- slower EMAs (100+),
- add a regime filter (only trade longs while close > EMA-200).

__Metrics:__

__*Expectancy:*__
- = win_rate * avg_win + (1 - win_rate) * avg_loss
- Folds win size and loss size into one number.
- Is sensitive to trade frequency.
- Meaning: the expected net P&L per trade in quote currency (USDT), after fees and slippage, weighted by win/loss probability.
- Question: if I take one more trade with this configuration, what's my expected dollar outcome?
- Example: expectancy = −2.52 means: across N trades the strategy loses $2.52 of net P&L per trade on average.
- Results:
    - Positive = the strategy makes money on average per trade.
    - Negative = the strategy loses money on average.

__*Profit factor:*__
- = (gross_win / gross_loss) if gross_loss > 0 else float("inf")
- Expectancy expressed as a ratio: PF > 1 ⟺ expectancy > 0.
- Is not sensitive to trade frequency; tells the edge per dollar at risk regardless of how often you trade.
- Meaning: the ratio of cumulative wins to cumulative losses; ratio of total dollars won to total dollars lost across all trades.
- Question: For every $1 the strategy lost, how many dollars did it win back?
- Results:
    - < 1.0 unprofitable
    - 1.0–1.3 marginal, easily eaten by a regime change or fee bumps. PF = 0.50 means you win $0.50 for every $1 lost.
    - 1.3–1.7 decent, tradeable
    - 1.7–3.0 strong. PF = 2.0 means you win $2 for every $1 lost.
    - .> 3.0 suspiciously good — usually overfitting or a small sample

__*Max drawdown %:*__
- = (equity_curve - equity_curve.cummax()) / equity_curve.cummax() * 100.min()
- Captures the worst point of the worst losing streak.
- Reflects how sustainable a strategy is in terms of risk of ruin. 
- Two strategies can have identical final P&L but very different max DDs. In live trading, positions are often closed during a 33% drawdown before any recovery occurs. 
- Meaning: the deepest peak-to-trough decline of the equity curve, expressed as a percent of the prior peak.
- Question: What's the worst paper loss I'd have lived through to keep running this strategy?
- Results:
    - A common practice is to assume the live max DD will be 1.3–2× the backtested max DD, especially for strategies with negative or low expectancy.
    - It's a single point in time — doesn't tell you how long the drawdown lasted or how many times you suffered a similar one.

__*Sharpe:*__
- = (returns.mean() / returns.std() * np.sqrt(252)) if returns.std() > 1e-4 else 0.0
- Is used here for ranking configs against each other within this codebase. 
- Is built  around daily per-trade returns, not annualized -> is not valid for comparing with industry benchmarks.
- Meaning: risk-adjusted return. How much return did you earn per unit of risk? 
- Question: Per unit of overall return variability, how much edge?

__*Sortino:*__
- = (returns.mean() / downside.std() * np.sqrt(252)) if len(downside) > 1 and downside.std() > 1e-4 else 0.0, with downside = returns[returns < 0]
- Is used here for ranking configs against each other within this codebase.
- Is built  around daily per-trade returns, not annualized -> is not valid for comparing with industry benchmarks.
- Meaning: risk-adjusted return. How much return did you earn per unit of risk? 
- Question: Per unit of downside pain, how much edge?
- Results: 
    - Sharpe ≈ Sortino → wins and losses have similar size distributions (typical for fixed-stop / fixed-target strategies).
    - Sortino > Sharpe → upside is fatter-tailed than downside (classic for trend-following: many small losses, occasional big winners).
    - Sortino < Sharpe → downside is fatter-tailed (mean-reversion sold without a stop, vol selling — the "picking up nickels in front of a steamroller" profile).

### Parameter sweep: stop × target

Logical flow: set up the grid → run the sweep → pick by train, judge on test → look at the leaderboard → interpret the tables.

Run a small grid over (stop%, R-target) on **whichever cfg you assembled above** — Route A's chosen candidate or Route B's train-picked EMA. The sweep takes cfg as a base config and only varies the stop and target.

Why grid these two together: a wider stop and a smaller R-target both push toward higher win-rate, lower-magnitude trades. We want to see whether either tweak — or the combination — turns expectancy positive after fees.

If you came from Route B and want a clean train→test rank-agreement read, run the sweep on train_df and test_df separately (the cell below does that). If you came from Route A, just sweep on the full df instead — change train_df / test_df to df in the calls.

`sweep_configs(df, base_cfg, stop_pcts, r_targets)` runs the full grid and returns a tidy DataFrame ranked by expectancy, with exit-reason counts inline (sl / tp / trail / eod).

__Run the grid on train and test__

In [ ]:
stop_pcts = [1.0, 1.5]
r_targets = [2.0, 3.0]

sweep_train = sweep_configs(train_df, cfg, stop_pcts, r_targets)
sweep_test  = sweep_configs(test_df,  cfg, stop_pcts, r_targets)

print("=== TRAIN ===")
print(sweep_train.to_string(index=False))
print("\n=== TEST ===")
print(sweep_test.to_string(index=False))


__How to read the table:__

- Each row is one full backtest with a different (stop_pct, r_target). Rows are sorted by expectancy (highest first).
- sl/tp/trail/eod are exit-reason counts — use them to sanity-check that more trades are reaching TP at lower R-targets and wider stops, as expected.
- The decision is: **does any row have positive expectancy on the test slice, and is its train rank similar?** If train ranks A > B > C > D and test ranks roughly agree, the ordering is real signal. If test re-shuffles randomly, the differences are noise.

__Result:__

The 1.5%/3R config was the worst on train (rank 4) and best on test (rank 1, the only positive). \
That's the opposite of the train→test agreement you want. \
With only 15 trades on test, this is almost certainly noise rather than signal. \
A truly robust edge would rank similarly on both slices.

__What this actually tells you:__

1. None of the four configs has a real edge in this 3-month window.
2. The fact that a bad-on-train config "won" on test is exactly why walk-forward exists — it stops you from deploying the 1.5%/3R variant just because the test number looks good.
3. To find a real edge, expand the grid: more EMA periods, slower timeframes, sweep direction (long / short / both), get more data (12+ months), or change the entry signal entirely (the touch+rejection logic may not be enough).

In [ ]:
best_test = sweep_test.iloc[0]
print(f"Best on test:  stop={best_test['stop_pct']}%   target={best_test['r_target']}R")
print(f"  expectancy={best_test['expectancy']}   profit_factor={best_test['profit_factor']}   return={best_test['total_return_pct']}%")

from dataclasses import replace
best_cfg = replace(
    cfg,
    stop_loss=StopLossConfig(mode="fixed_pct", value=float(best_test["stop_pct"])),
    take_profit=TakeProfitConfig(mode="r_multiple", value=float(best_test["r_target"])),
)
trades_best, equity_best = backtest(test_df, best_cfg)
print()
print_metrics(compute_metrics(trades_best, equity_best, best_cfg.initial_capital))


In [ ]:
plot_equity_curve(equity_best, best_cfg.initial_capital,
                  title=f"Equity — test, stop={best_cfg.stop_loss.value}% / target={best_cfg.take_profit.value}R")


In [ ]:
plot_trades_on_chart(test_df, trades_best, best_cfg, initial_window_days=30)

### Full sweep: stop × target × EMA × regime × direction

Logical flow: set up the grid → run the sweep → pick by train, judge on test → look at the leaderboard → interpret the tables.

Extend the grid to all five axes: every combination of (stop_pct, r_target, ema_period, regime_filter, direction).

- __EMA period__: include slower EMAs (50, 100, 150, 200) — the "respected" levels traders actually watch.
- __Regime filter__: None = no filter; an integer N = only enter longs while close > EMA-N (or only shorts while close < EMA-N).
  - Caveat: when the regime EMA equals or is faster than the entry EMA, the filter becomes redundant or near-redundant. The interesting cases are entry EMA ≥ regime EMA (e.g. entry on EMA-100 retrace, gated by EMA-200 regime).
- __Direction__: "long", "short", or "both". Sweeping this lets the data tell you whether the period is dominated by uptrend pullbacks, downtrend bounces, or balanced mean-reversion.
- The grid in the example below is 3 × 3 × 5 × 2 × 3 = __270 backtests__ per slice. All run in seconds on a few thousand candles.

We refit nothing on the test slice; we just rank the same configs on both train and test and look for configs whose ranks agree.


__Run the 270-combo grid__


Keep regime_filter ≥ max(ema_period) (or None), otherwise you get NaN rows when a combo has regime_filter < ema_period.

The same rule holds for shorts: a regime EMA much faster than the entry EMA blocks almost all entries. In a downtrend, price pulling UP to a slow entry EMA is by definition above any faster EMA, so close < regime_EMA fails — and shorts never fire.

In [ ]:
grid = {
    "stop_pct":       [0.5, 1.0, 1.5],
    "r_target":       [2.0, 2.5, 3.0],
    "ema_period":     [20, 50, 54, 115, 118],
    "regime_filter":  [None, 200],
    "direction":      ["long", "short", "both"],
}

full_sweep_train = sweep_grid(train_df, cfg, grid)
full_sweep_test  = sweep_grid(test_df,  cfg, grid)

print("=== TRAIN — top 10 by expectancy ===")
print(full_sweep_train.head(10).to_string(index=False))
print("\n=== TEST — top 10 by expectancy ===")
print(full_sweep_test.head(10).to_string(index=False))


__How to read the result:__

Look for a config that's near the top on train AND still positive on test. The shape that often wins on chop-heavy slices: very slow EMA (e.g. 200) + macro filter (entry only when close on the right side of a slower EMA) + wider stop + asymmetric target — effectively "trade retracements to a major dynamic level only when the macro trend agrees."

__Caveats:__

- A train→test agreement with <20 trades on test is tiny — a few extra wins or losses can flip the ranking. Treat as a hypothesis, not a strategy.
- A 270-combo sweep on a 3-month window is statistically suspect (multiple-hypothesis problem — with 270 configs, some will look great by chance).

__Further actions:__

1. Re-run on 12+ months of data. With ~10× the trade count, a true edge will solidify or fall apart.
2. Add adjacent EMA periods around the winner — if the edge is real, neighboring EMAs should also rank well.
3. Test on ETH and a couple of other liquid majors — a true edge generalizes.


__Pick by train expectancy, judge on test__

The honest workflow: pick the best config *as ranked on train*, then look up that exact config's row on test. If the train-best is also profitable on test, you have a candidate edge. If it collapses on test, the apparent edge was overfit.

In [ ]:
# Best on TRAIN (the only slice you're allowed to pick from)
key_cols = ["stop_pct", "r_target", "ema_period", "regime_filter", "direction"]
train_pick = full_sweep_train.iloc[0]
print("Train pick:")
print(train_pick[key_cols + ["trades", "win_rate", "expectancy", "profit_factor", "total_return_pct"]].to_string())

# Find that exact config on test
match = full_sweep_test
for k in key_cols:
    v = train_pick[k]
    if pd.isna(v):
        match = match[match[k].isna()]
    else:
        match = match[match[k] == v]
print()
print("Same config on test:")
if len(match) == 0:
    print("  (no matching row — should not happen with shared grid)")
else:
    print(match.iloc[0][key_cols + ["trades", "win_rate", "expectancy", "profit_factor", "total_return_pct"]].to_string())


__Top-3 configs on test__ (for context — these are NOT picks, just the leaderboard)

In [ ]:
full_sweep_test.head(3)[key_cols + ["trades", "win_rate", "expectancy", "profit_factor", "total_return_pct", "max_dd_pct", "sl_n", "tp_n"]]


__How to read these tables:__

- __Look for rank agreement, not point estimates.__ A config that's rank-2 on train and rank-4 on test is a stronger signal than one that's rank-25 on train and rank-1 on test (the latter is overfit-luck).
- __Check trades per row.__ With ~6000 train candles, a 50-EMA setup with regime filter might produce 30–60 trades; a 200-EMA with filter might produce <10. Anything with <20 trades is a noisy estimate.
- __Sample-size adjustment__: when comparing test rows, prefer configs with more trades — small trades makes expectancy unstable.
- __Regime filter check__: compare otherwise-identical rows with regime_filter=None vs a set value. The filter should reduce trade count and *might* improve win rate. If win rate doesn't move, the filter isn't doing anything useful at the chosen EMAs.
- __Direction balance (when sweeping direction='both')__: a healthy both run shows roughly comparable long_trades and short_trades counts on a chop-heavy slice; a lopsided count means one side is being filtered out by the regime/touch conditions for most of the period — check whether you'd have done better picking that side explicitly.

__If nothing positive on test:__ this dataset (3 months) probably doesn't contain enough variety. Re-run with 12+ months of history; the longer window will both stabilize statistics and let you walk forward across regime changes.